In [ ]:

# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,201)

resolution_width=np.ones_like(Q)*0.1+0.005*np.random.normal(size=len(Q))

diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(Q),len(E)))

scale=0.7 #arbitrary scale factor for diffusion model

T=3

model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
HWHM=model.calculate_width(Q)

QQISF=model.calculate_QISF(Q)
EISF=model.calculate_EISF(Q)


resolution_handler=ResolutionHandler()

sample_model=[]
for i in range(len(Q)):
    sample_model.append(SampleModel(name=f"SampleModel_{i}"))

    sample_model[i].add_component(DeltaFunction(area=scale*EISF[i]+0.23, name="Elastic"))
    sample_model[i].add_component(Lorentzian(area=scale*QQISF[i], name="QuasiElastic", width=HWHM[i]) )
    resolution=Gaussian(name="Resolution", area=1,width=resolution_width[i])

    convoluted_signal[i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.05+0.01*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Q','energy'],values=convoluted_signal,variances=0.01*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp})

